In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path('/content/drive/MyDrive/Brain_alignment/lpp_data')

# Load GPT-2 TR embeddings from Day 2
gpt2 = np.load(DATA / 'gpt2_tr_embeddings_run15.npz')
print("Context conditions:", list(gpt2.keys()))
print("Shape per condition:", gpt2['1'].shape)  # (286, 1024)

Mounted at /content/drive
Context conditions: ['1', '5', '20', '50', '100', 'full']
Shape per condition: (286, 1024)


In [ ]:
import nibabel as nib

img  = nib.load(str(DATA / 'sub-EN057/func/sub-EN057_task-lppEN_run-15_echo-1_bold.nii.gz'))
bold = img.get_fdata()  # (64, 64, 33, 286)

# Reshape to (n_TRs, n_voxels)
X, Y, Z, T = bold.shape
bold_flat = bold.reshape(-1, T).T  # (286, 139264)

# Brain mask: keep voxels with non-zero variance
mask = bold_flat.std(axis=0) > 0
bold_masked = bold_flat[:, mask]  # (286, n_voxels)
print(f"Brain voxels after masking: {mask.sum()} / {len(mask)}")
print(f"BOLD matrix: {bold_masked.shape}")

Brain voxels after masking: 128926 / 135168
BOLD matrix: (286, 128926)


In [ ]:
from scipy.signal import fftconvolve

def hrf(tr=2.0, n_trs=30):
    """Double-gamma HRF sampled at TR resolution."""
    from scipy.stats import gamma
    t = np.arange(0, n_trs * tr, tr)
    h = (gamma.pdf(t, 6) - 0.35 * gamma.pdf(t, 16))
    return h / h.max()

def convolve_hrf(X, tr=2.0):
    """Convolve each feature column with HRF."""
    h = hrf(tr=tr, n_trs=20)
    out = np.zeros_like(X)
    for i in range(X.shape[1]):
        conv = fftconvolve(X[:, i], h, mode='full')
        out[:, i] = conv[:X.shape[0]]
    return out

# Convolve all context conditions with HRF
gpt2_hrf = {}
for ctx in gpt2.keys():
    gpt2_hrf[ctx] = convolve_hrf(gpt2[ctx])
    print(f"  context={ctx}: convolved shape {gpt2_hrf[ctx].shape}")

  context=1: convolved shape (286, 1024)
  context=5: convolved shape (286, 1024)
  context=20: convolved shape (286, 1024)
  context=50: convolved shape (286, 1024)
  context=100: convolved shape (286, 1024)
  context=full: convolved shape (286, 1024)


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

def encode_model_pca(X, Y, n_components=50, n_splits=5,
                     alphas=(0.01, 0.1, 1, 10, 100, 1000)):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Reduce dimensionality: 1024 → 50
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)
    print(f"  PCA variance explained: {pca.explained_variance_ratio_.sum():.2f}")

    kf = KFold(n_splits=n_splits, shuffle=False)
    r2_folds = []

    for train_idx, test_idx in kf.split(X_pca):
        X_tr, X_te = X_pca[train_idx], X_pca[test_idx]
        Y_tr, Y_te = Y[train_idx],     Y[test_idx]

        ridge = RidgeCV(alphas=alphas)
        ridge.fit(X_tr, Y_tr)
        Y_pred = ridge.predict(X_te)

        r2 = r2_score(Y_te, Y_pred, multioutput='raw_values')
        r2_folds.append(r2)

    return np.mean(r2_folds, axis=0)

r2_scores = {}
for ctx in gpt2_hrf.keys():
    print(f"Running context={ctx}...")
    r2_scores[ctx] = encode_model_pca(gpt2_hrf[ctx], bold_masked)
    pos = r2_scores[ctx] > 0
    print(f"  Mean R² (positive): {r2_scores[ctx][pos].mean():.4f}, "
          f"positive voxels: {pos.sum()} ({100*pos.mean():.1f}%)")

Running context=1...
  PCA variance explained: 0.82
  Mean R² (positive): 0.0183, positive voxels: 147 (0.1%)
Running context=5...
  PCA variance explained: 0.81
  Mean R² (positive): 0.0166, positive voxels: 157 (0.1%)
Running context=20...
  PCA variance explained: 0.80
  Mean R² (positive): 0.0166, positive voxels: 157 (0.1%)
Running context=50...
  PCA variance explained: 0.80
  Mean R² (positive): 0.0179, positive voxels: 161 (0.1%)
Running context=100...
  PCA variance explained: 0.80
  Mean R² (positive): 0.0148, positive voxels: 173 (0.1%)
Running context=full...
  PCA variance explained: 0.81
  Mean R² (positive): 0.0162, positive voxels: 183 (0.1%)


In [ ]:
# Save R² scores (in masked space) + mask
np.savez(DATA / 'r2_scores_run15.npz',
         mask=mask,
         **{f'ctx_{k}': v for k, v in r2_scores.items()})
print("Saved: r2_scores_run15.npz")

# Quick summary table
print("\n=== Brain Alignment by Context Size ===")
for ctx in ['1', '5', '20', '50', '100', 'full']:
    r2 = r2_scores[ctx]
    pos = r2 > 0
    print(f"  context={ctx:>4}: mean R²={r2[pos].mean():.4f}, "
          f"positive voxels={pos.sum()} ({100*pos.mean():.1f}%)")

Saved: r2_scores_run15.npz

=== Brain Alignment by Context Size ===
  context=   1: mean R²=0.0183, positive voxels=147 (0.1%)
  context=   5: mean R²=0.0166, positive voxels=157 (0.1%)
  context=  20: mean R²=0.0166, positive voxels=157 (0.1%)
  context=  50: mean R²=0.0179, positive voxels=161 (0.1%)
  context= 100: mean R²=0.0148, positive voxels=173 (0.1%)
  context=full: mean R²=0.0162, positive voxels=183 (0.1%)


In [ ]:
# Sanity check: does a shuffled predictor give lower R² than real?
from sklearn.utils import shuffle

ctx = 'full'
X_real    = gpt2_hrf[ctx]
X_shuffle = shuffle(X_real, random_state=42)

r2_real  = encode_model_pca(X_real,    bold_masked)
r2_shuff = encode_model_pca(X_shuffle, bold_masked)

print(f"Real    — mean R²: {r2_real.mean():.5f},    positive voxels: {(r2_real>0).sum()}")
print(f"Shuffled — mean R²: {r2_shuff.mean():.5f}, positive voxels: {(r2_shuff>0).sum()}")

  PCA variance explained: 0.81
  PCA variance explained: 0.81
Real    — mean R²: -1.52304,    positive voxels: 211
Shuffled — mean R²: -2.86838, positive voxels: 98


In [ ]:
# Define the path to the functional files
FUNC = DATA / 'sub-EN057/func'

# Load word_info if it hasn't been loaded yet
# (Adjust the filename if your CSV has a different name in the lpp_data folder)
word_info_path = DATA / 'annotation/EN/lppEN_word_information.csv'
if word_info_path.exists():
    word_info = pd.read_csv(word_info_path)
else:
    # Fallback or placeholder in case the path differs
    word_info = pd.DataFrame(columns=['section', 'offset'])
    print(f"Warning: Could not find word_info at {word_info_path}. Please adjust the path.")

run_nums = list(range(15, 24))
sections = list(range(1, 10))

print("Run | N_TRs | Duration | Best section match")
print("-" * 55)
for run, sec in zip(run_nums, sections):
    bold_path = FUNC / f'sub-EN057_task-lppEN_run-{run}_echo-1_bold.nii.gz'
    if bold_path.exists():
        img = nib.load(str(bold_path))
        n_trs = img.shape[3]
        duration = n_trs * 2.0

        # Retrieve matching section info
        sec_rows = word_info[word_info['section'] == sec]
        if not sec_rows.empty:
            sec_dur = sec_rows['offset'].max()
            diff = abs(sec_dur - duration)
            print(f"Run {run} → Section {sec}: {n_trs} TRs = {duration:.0f}s, "
                  f"section max offset = {sec_dur:.1f}s, diff = {diff:.1f}s")
        else:
            print(f"Run {run} → Section {sec}: {n_trs} TRs = {duration:.0f}s (No matching section info in word_info)")

Run | N_TRs | Duration | Best section match
-------------------------------------------------------
Run 15 → Section 1: 286 TRs = 572s, section max offset = 554.6s, diff = 17.4s
Run 16 → Section 2: 302 TRs = 604s, section max offset = 587.7s, diff = 16.3s
Run 17 → Section 3: 344 TRs = 688s, section max offset = 671.3s, diff = 16.7s
Run 18 → Section 4: 307 TRs = 614s, section max offset = 598.2s, diff = 15.8s


In [11]:
import nibabel as nib
import pandas as pd
import numpy as np
from pathlib import Path

DATA      = Path('/content/drive/MyDrive/Brain_alignment/lpp_data')
FUNC      = DATA / 'sub-EN057/func'
ANNOT     = DATA / 'annotation/EN'
word_info = pd.read_csv(ANNOT / 'lppEN_word_information.csv')

RUN_SECTION_MAP = {15: 1, 16: 2, 17: 3, 18: 4}

alignments = {}
bold_runs  = {}

for run, sec in RUN_SECTION_MAP.items():
    # Load BOLD
    img    = nib.load(str(FUNC / f'sub-EN057_task-lppEN_run-{run}_echo-1_bold.nii.gz'))
    n_trs  = img.shape[3]
    bold_runs[run] = img.get_fdata()

    # Build alignment
    sec_words = word_info[word_info['section'] == sec].copy()
    sec_words['TR_index'] = (sec_words['onset'] / 2.0).astype(int).clip(upper=n_trs - 1)
    alignments[run] = sec_words

    print(f"Run {run} (section {sec}): {n_trs} TRs, {len(sec_words)} words, "
          f"TR range {sec_words['TR_index'].min()}–{sec_words['TR_index'].max()}")

# Save corrected alignment for run 15 / section 1
alignments[15].to_csv(DATA / 'word_TR_alignment_run15.csv', index=False)
print("\nCorrected run15 alignment saved.")

Run 15 (section 1): 286 TRs, 1517 words, TR range 0–277
Run 16 (section 2): 302 TRs, 1690 words, TR range 0–293
Run 17 (section 3): 344 TRs, 1852 words, TR range 0–335
Run 18 (section 4): 307 TRs, 1624 words, TR range 0–298

Corrected run15 alignment saved.
